# Modern Automated Thai GEC Fine-Tuning 🇹🇭🚀
This notebook uses the modern, industry-standard Hugging Face stack (`trl`, `peft`, `bitsandbytes`, `accelerate`) to fine-tune LLMs on Thai Grammatical Error Correction data.
It is robust, stable on Google Colab, and does not require DeepSpeed for 7B/13B models on an A100.


In [ ]:
# ==========================================
# ⚙️ CONFIGURATION & PARAMETERS
# ==========================================

# 1. Experiment Type
# Options: "FT", "SI", "SI-150%", "SI-200%", "SI-300%"
EXPERIMENT_TYPE = "SI-150%" 

# 2. Model Selection
MODEL_ID = "scb10x/typhoon-7b"

# 3. Training Hyperparameters
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 2
WARMUP_STEPS = 100
SAVE_STEPS = 500
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048

# 4. Credentials & Hub
HUGGINGFACE_TOKEN = "your_hf_token_here" # Get from https://huggingface.co/settings/tokens
HUB_USERNAME = "pakawadeep"
PUSH_TO_HUB_WHILE_TRAINING = True

# Hub Model Names
LORA_HUB_MODEL_ID = f"{HUB_USERNAME}/{MODEL_ID.split('/')[-1]}-lora-ctfl-th-gec-{EXPERIMENT_TYPE}"
HUB_MODEL_ID = f"{HUB_USERNAME}/{MODEL_ID.split('/')[-1]}-ctfl-th-gec-{EXPERIMENT_TYPE}"

# 5. Data Paths (Google Drive Mounted)
TRAIN_PATH = '/content/drive/MyDrive/NLP Thesis/Data/in-use/train-ctfl-cleaned.csv'
GENERATED_PATH = '/content/drive/MyDrive/NLP Thesis/Data/generated_instruction.csv'


## 1. Installation & Setup

In [ ]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes hf_transfer


In [ ]:
from huggingface_hub import login
if HUGGINGFACE_TOKEN and HUGGINGFACE_TOKEN != "your_hf_token_here":
    login(token=HUGGINGFACE_TOKEN)
else:
    print("Warning: Hugging Face token not provided or left as default.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save Hugging Face cache to Google Drive so models are never redownloaded!
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/HF_Cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


## 2. Data Preprocessing

In [ ]:
import pandas as pd
from datasets import Dataset

# Load train dataset
df_train = pd.read_csv(TRAIN_PATH)
print(f"Original FT rows: {len(df_train)}")

# Mix generated data based on experiment type
if EXPERIMENT_TYPE != "FT":
    df_gen = pd.read_csv(GENERATED_PATH)
    
    if EXPERIMENT_TYPE == "SI":
        df_gen = df_gen.sample(n=4843, random_state=42)
    elif EXPERIMENT_TYPE == "SI-150%":
        df_gen = df_gen.sample(n=7018, random_state=42)
    elif EXPERIMENT_TYPE == "SI-200%":
        df_gen = df_gen.sample(n=10004, random_state=42)
    elif EXPERIMENT_TYPE == "SI-300%":
        pass # All rows
        
    df_combined = pd.concat([df_train, df_gen], ignore_index=True)
else:
    df_combined = df_train

print(f"Total training rows for {EXPERIMENT_TYPE}: {len(df_combined)}")

# Shuffle the combined dataset
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

def format_data(example):
    defaultPrompt = '''แก้ไขประโยคต่อไปนี้ให้มีการใช้ภาษาที่ถูกต้อง โดยแก้ไขเฉพาะที่จำเป็น'''
    formatted = f"### Instruction: {defaultPrompt}\n\n### Input: {example['text']}\n\n### Output: {example['expected']}"
    return {"text": formatted}

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df_combined)
dataset = dataset.map(format_data)

print("Sample:")
print(dataset[0]['text'])


## 3. Training Setup (QLoRA)

In [ ]:
# Download the model robustly using the Hugging Face CLI
# This bypasses the Python progress bar issues that often freeze Colab downloads.
!huggingface-cli download {MODEL_ID} --local-dir ./local_model_cache --resume-download

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Quantization Config (4-bit QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained("./local_model_cache", token=HUGGINGFACE_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for FP16 training issues

model = AutoModelForCausalLM.from_pretrained(
    "./local_model_cache", 
    quantization_config=bnb_config, 
    device_map="auto",
    token=HUGGINGFACE_TOKEN
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    optim="paged_adamw_32bit",
    save_steps=SAVE_STEPS,
    logging_steps=10,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=WARMUP_STEPS,
    group_by_length=True,
    lr_scheduler_type="cosine",
    push_to_hub=PUSH_TO_HUB_WHILE_TRAINING,
    hub_model_id=LORA_HUB_MODEL_ID,
    hub_token=HUGGINGFACE_TOKEN,
    hub_private_repo=True,
    report_to="none" # Set to "wandb" if you want to use W&B
)

# 5. SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)


## 4. Run Training

In [ ]:
# Start Training!
trainer.train()

# Push the final LoRA adapter to Hub
if PUSH_TO_HUB_WHILE_TRAINING:
    trainer.push_to_hub()


## 5. Fuse & Push Full Model

In [ ]:
# ==========================================
# 🧩 FUSING MODEL & PUSHING FULL WEIGHTS
# ==========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gc

# Free VRAM before merging
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading base model in FP16...")
base_model = AutoModelForCausalLM.from_pretrained(
    "./local_model_cache",
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HUGGINGFACE_TOKEN
)

print(f"Loading LoRA adapters from {LORA_HUB_MODEL_ID}...")
model_to_merge = PeftModel.from_pretrained(
    base_model, 
    LORA_HUB_MODEL_ID, 
    token=HUGGINGFACE_TOKEN
)

print("Merging weights...")
merged_model = model_to_merge.merge_and_unload()

print(f"Pushing merged model to {HUB_MODEL_ID}...")
merged_model.push_to_hub(HUB_MODEL_ID, private=True, token=HUGGINGFACE_TOKEN)
tokenizer.push_to_hub(HUB_MODEL_ID, private=True, token=HUGGINGFACE_TOKEN)

print("✅ Fusing and upload complete!")
